# BE RMI - RADCLIM: Radar–Rain-Gauge Merged Precipitation over Belgium

This notebook describes how to access and use the zarr-version of the **[BE RADCLIM 1 km Composite Rainfall](https://opendata.meteo.be/)** dataset, produced by the Royal Meteorological Institute of Belgium (RMI). RADCLIM is an offline, reprocessed quantitative precipitation dataset designed for climatological analysis, filling in gaps that occurred in the real-time RADQPE product.

The dataset provides **1 km gridded precipitation** accumulations on a 700x700 grid over Belgium and the surrounding area (0.3°W–9.7°E, 47.4–53.7°N) at **5-minute rain-rate** (mm/h)

Precipitation fields are derived from a network of **four C-band (partially dual-polarisation) weather radars** (Jabbeke, Wideumont, Helchteren, Avenois) covering the Belgian territory. For this product, radar estimates are merged with rain-gauge observations from automatic weather stations of multiple Belgian networks (excluding climatological stations) using **mean-field bias (MFB) correction**.

The `v1.0.0` zarr version of the dataset covers **2017-01-01 to 2023-11-30** (archive period). It is projected in the **Belgian Lambert 2008** coordinate reference system (EPSG:3812).

License: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)

In [ ]:
import matplotlib.pyplot as plt
import mlcast_datasets

In [ ]:
# Open the mlcast datasets intake catalog and list available precipitation datasets
cat = mlcast_datasets.open_catalog()
list(cat.precipitation)

## 5-minute radar-rain-gauge merged rain rate

The RADCLIM 5-minute rain-rate product with mean-field bias (MFB) correction is available in the intake catalog as `be_rmi_radclim_mfb_5min`. 

In [ ]:
# Load the RADCLIM 5-minute rain-rate dataset lazily via Dask
ds = cat.precipitation.be_rmi_radclim_mfb_5min.to_dask()
ds

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pyproj import CRS


var_name = "rain_rate"
crs_name = ds[var_name].grid_mapping
data_crs = ccrs.Projection(ds[crs_name].crs_wkt)

In [ ]:
# Plot three consecutive 5-minute timesteps for a selected date
# The mid-July 2021 event was a historically severe flooding event over Belgium
# (see Journée et al., 2023 in the References section below)
g = (
    ds[var_name]
    .sel(time="2021-07-14T16")
    .isel(time=slice(None, 3))
    .plot(
        transform=data_crs,
        cmap="viridis",
        add_colorbar=True,
        col="time",
        robust=True,
        subplot_kws=dict(projection=data_crs),
        vmin=0,
    )
)

for ax in g.axes.flat:
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linewidth=0.8, edgecolor="black")
    ax.gridlines(draw_labels=["top", "left"])

## 📚 References & Further Reading

The RADCLIM dataset is based on the RADQPE methodology. The following publications describe the underlying algorithms and a notable application of the dataset:

- Goudenhoofdt, E. and Delobbe, L. (2016), *"Generation and Verification of Rainfall Estimates from 10-Yr Volumetric Weather Radar Measurements"*, **Journal of Hydrometeorology**, Vol. 17, Issue 4, pp. 1223–1242.
  → [https://doi.org/10.1175/JHM-D-15-0166.1](https://doi.org/10.1175/JHM-D-15-0166.1)

- Journée, M., Goudenhoofdt, E., Vannitsem, S. and Delobbe, L. (2023), *"Quantitative rainfall analysis of the 2021 mid-July flood event in Belgium"*, **Hydrology and Earth System Sciences**, Vol. 27, Issue 17, pp. 3169–3189.
  → [https://doi.org/10.5194/hess-27-3169-2023](https://doi.org/10.5194/hess-27-3169-2023)

---

**📬 Maintainer:** Royal Meteorological Institute of Belgium (RMI)  
**🔑 License:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)  
**📥 More information and documentation:** [RMI Opendata Server](https://opendata.meteo.be/)
**⚠️ Use Limitation:** This is a pre-release and can contain errors.